# 4. Forced response: figures

The figures for the forced response, drawn from what notebook 03 saved, so nothing slow runs here. Notebook 03 explains each method, step by step.

Every section shows its result in the same four figures:

1. **What goes in.** The quantities the result is made from, then the result: one model, experiment and season (`MODEL`, `EXPERIMENT`, `SEASON`).
2. **One model.** The parts and the result for every experiment of `MODEL` (one row each).
3. **Every model.** The result for every model (rows) and experiment (columns), dotted where it is not significant.
4. **How many models.** At each grid point, how many models have a significant increase (red, top row) and how many a significant decrease (blue, bottom row).

| Section | Result | Test |
|---|---|---|
| 1. Mean change | final 21 years against hist-nat's | member-block permutation test |
| 2. Signal to noise | S/N | \|S/N\| ≥ 2 |
| 3. Quantile response | the change in Q05, Q50 and Q95 | none |
| 4. Change in width | the change in Q95 − Q05, and which tail did it | hist-nat bootstrap |
| 5. The whole distribution | Q05, the mean, Q95 and the width together, and where the mean and the width changed (after Bracegirdle et al., 2024) | sections 1 and 4 |
| 6. Additivity | historical against the sum of the single forcings | none |
| 7. Zonal means and the sea-ice edge | sections 1 and 4 by latitude (after Bracegirdle et al., 2024) | sections 1 and 4, zonally averaged |

**Reading the figures.** A quantity has the same colour scale in every figure (set once, under *Colour scales*). Every change is red for an increase and blue for a decrease, dots mark where a change is not significant, and every map is of `SEASON`. The paper figures, which put the models together, are in notebook 06. The cells before each figure pick out its data and show them.

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame). Each saved result has the same name as in notebook 03. The previous version of this notebook is in `archive/forced_response_figures_2026-10-07.ipynb`.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from plotting_modules import formatting
from extant import config, loading, paths, preprocessing as pp, response_change as rc, sea_ice
from extant import significance as sig, storage, zonal
from extant.logger import setup_logging
from extant.plots import forced_response as fr, response_change as rcp, sea_ice as sip, zonal as zp

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

## Settings

In [ ]:
VARIABLE = 'tas'
var = config.VARIABLES[VARIABLE]
UNITS = var.units

WINDOW = config.WINDOW
LAST_FULL_YEAR = config.LAST_FULL_YEAR

#(t): The experiments, in the order of every figure, and the seasons
EXPERIMENTS = ['hist-GHG', 'hist-aer', 'hist-totalO3', 'historical']
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']

#(t): The model and experiment of the "what goes in" and "one model" figures, and the season of every map
MODEL = config.PLOT_SEL['model']
EXPERIMENT = config.PLOT_SEL['experiment']
SEASON = 'DJF'

#(t): Significance level of every test, and the |S/N| counted as emerged
ALPHA = 0.05
SN_THRESHOLD = 2

#(t): The quantiles of section 3, and how they are labelled
RESPONSE_QUANTILES = [0.05, 0.50, 0.95]
QUANTILE_LABELS = {0.05: 'Q05', 0.50: 'Q50', 0.95: 'Q95'}


def result_path(name):
    """Where notebook 03 saved a result: results/<variable>/<run>/<name>.zarr."""
    return paths.result(name, VARIABLE, RUN)

### Colour scales

One scale per quantity, used by every figure that shows it (here and in notebooks 05 and 06), so the same colour always means the same value. Each is a `fr.Scale`: the contour levels, the colormap and the colour-bar label. Their ranges, per variable, are in `fr.RANGES`; notebook 06 (*Check the colour ranges*) says how much of each field falls beyond them.

In [ ]:
scales = fr.colour_scales(VARIABLE, alpha=ALPHA, sn_threshold=SN_THRESHOLD)

In [ ]:
#(t): What each one covers
pd.DataFrame({name: {'from': scale.levels[0], 'to': scale.levels[-1], 'colormap': getattr(scale.cmap, 'name', scale.cmap),
                     'label': scale.label}
              for name, scale in vars(scales).items()}).T

## Open the data and the saved results

Run **one** of the next two sections. Both leave `lesfmip_season_tree` (the seasonal means, used only for the figures at one grid point in section 5) and `RUN`, which says which saved results to open.

### Sample data

In [ ]:
#(t): The sample's monthly means, as a DataTree /<model>/<experiment>
lesfmip_sample_tree = loading.open_lesfmip_sample(experiments=config.MAIN_EXPERIMENTS)
lesfmip_sample_tree

In [ ]:
#(t): Seasonal means in analysis units, through the same pre-processing as notebook 01
lesfmip_season_tree = pp.seasonal_tree(lesfmip_sample_tree, VARIABLE).compute()
RUN = 'sample'

### Full data (JASMIN)

In [ ]:
#(t): The seasonal means notebook 01 saved: DATA_DIR is read first, then SCRATCH
seasonal_path = paths.find(paths.seasonal('lesfmip', VARIABLE))
seasonal_path

In [ ]:
lesfmip_season_tree = xr.open_datatree(seasonal_path, engine='zarr', chunks={})
RUN = 'full'

In [ ]:
#(t): Only the experiments analysed (filter keeps the model nodes above them)
lesfmip_season_tree = lesfmip_season_tree.filter(lambda node: node.name in config.MAIN_EXPERIMENTS)
lesfmip_season_tree

### The saved results

Everything notebook 03 computed, from `results/<variable>/<RUN>/` under `paths.DATA_DIR` or `paths.SCRATCH` (the log gives each full path), loaded into memory. Grouped by the section that uses them.

In [ ]:
#(t): Section 1: every experiment's ensemble mean over its final WINDOW years (hist-nat's too), and the member-block test
final_mean_ds = storage.open_dataset(result_path('final_mean'), load=True)
mean_block_ds = storage.open_dataset(result_path('mean_block'), load=True)

#(t): Section 2: the forced responses, each experiment's noise, and the signal, noise and S/N made from them
lesfmip_forced_ds = storage.open_dataset(result_path('forced'), load=True)
lesfmip_experiment_noise_ds = storage.open_dataset(result_path('experiment_noise'), load=True)
lesfmip_signal_ds = storage.open_dataset(result_path('signal'), load=True)
lesfmip_noise_ds = storage.open_dataset(result_path('noise'), load=True)
lesfmip_seasonal_sn_ds = storage.open_dataset(result_path('sn'), load=True)

#(t): Section 3: hist-nat's whole-record quantiles, every experiment's final-years quantiles, and the change
q_histnat_base_ds = storage.open_dataset(result_path('q_histnat_base'), load=True)
final_q_da = storage.open_dataarray(result_path('final_quantiles'), load=True)
quantile_response_da = storage.open_dataarray(result_path('quantile_response'), load=True)
quantile_curve_da = storage.open_dataarray(result_path('quantile_curve'), load=True)

#(t): Section 4: the quantiles of the internal variability, the width test, and the tails
width_quantiles_ds = storage.open_dataset(result_path('width_quantiles'), load=True)
qrange_change_ds = storage.open_dataset(result_path('qrange_change'), load=True)
tails_ds = storage.open_dataset(result_path('tails'), load=True)

#(t): Sections 6 and 7: each experiment's change from its own baseline, and the zonal-mean tests
own_changes_da = storage.open_dataarray(result_path('own_changes'), load=True)
zonal_mean_ds = storage.open_dataset(result_path('zonal_mean_test'), load=True)
zonal_width_ds = storage.open_dataset(result_path('zonal_width_test'), load=True)

In [ ]:
#(t): What each result holds: its dims and its variables
results = {
    'final_mean_ds': final_mean_ds, 'mean_block_ds': mean_block_ds, 'lesfmip_forced_ds': lesfmip_forced_ds,
    'lesfmip_experiment_noise_ds': lesfmip_experiment_noise_ds, 'lesfmip_signal_ds': lesfmip_signal_ds,
    'lesfmip_noise_ds': lesfmip_noise_ds, 'lesfmip_seasonal_sn_ds': lesfmip_seasonal_sn_ds,
    'q_histnat_base_ds': q_histnat_base_ds, 'final_q_da': final_q_da, 'quantile_response_da': quantile_response_da,
    'quantile_curve_da': quantile_curve_da, 'width_quantiles_ds': width_quantiles_ds,
    'qrange_change_ds': qrange_change_ds, 'tails_ds': tails_ds, 'own_changes_da': own_changes_da,
    'zonal_mean_ds': zonal_mean_ds, 'zonal_width_ds': zonal_width_ds,
}
pd.DataFrame({
    name: {'dims': dict(result.sizes), 'variables': ', '.join(result.data_vars) if isinstance(result, xr.Dataset) else ''}
    for name, result in results.items()
}).T

In [ ]:
#(t): The number of models: the top of the colour bar of every count figure
N_MODELS = mean_block_ds.sizes['model']
N_MODELS

### Sea ice

The sea-ice edge on the zonal-mean figures and maps of section 7 comes from notebook 01, section 5. Run **one** of the next two sections. Both leave `ice_extents_tree` (each member's seasonal extent), `ice_climatology_da` (the ensemble-mean concentration over the final years) and `ice_edges_da` (the equivalent latitude of the edge in the final years); without sea ice they are `None` and the figures leave the edge out.

#### Without sea ice

For the sample (which has none), or before notebook 01 has saved it.

In [ ]:
ice_extents_tree = ice_climatology_da = ice_edges_da = None

#### With sea ice (JASMIN)

In [ ]:
ice_extents_tree = storage.open_tree(paths.result('extent', 'siconc', 'full'), load=True)
ice_climatology_da = storage.open_dataarray(paths.result('climatology', 'siconc', 'full'), load=True)
ice_edges_da = sea_ice.edge_latitude(ice_extents_tree, years=WINDOW)

## Check the results

The headline changes of every model and experiment in `SEASON`, averaged over the whole area (cos-latitude weighted). A model whose numbers sit far from the others' has probably gone wrong before notebook 03 (in its raw files or their conversion), and every figure below would show it; look at its members in notebook 03 before reading anything into it.

In [ ]:
#(t): The headline changes side by side: the mean, three quantiles, the width, and S/N in the centre of the final years
headline_ds = xr.Dataset({
    'mean change': mean_block_ds['mean_change'],
    'ΔQ05': quantile_response_da.sel(quantile=0.05, drop=True),
    'ΔQ50': quantile_response_da.sel(quantile=0.50, drop=True),
    'ΔQ95': quantile_response_da.sel(quantile=0.95, drop=True),
    'Δ(Q95 − Q05)': qrange_change_ds['qrange_change'],
    'S/N': lesfmip_seasonal_sn_ds[VARIABLE].isel(year=LAST_FULL_YEAR, drop=True),
})
headline_ds = headline_ds.sel(experiment=EXPERIMENTS, season=SEASON)
headline_ds

In [ ]:
#(t): Their area means, one row per model and experiment
headline_area_ds = sig.area_mean(headline_ds)
headline_area_df = headline_area_ds.to_dataframe().round(2)
headline_area_df

## 1. Mean change

The ensemble mean of each experiment's final 21 years minus hist-nat's over the same years (notebook 03, section 4):

$$
\Delta\bar{x} = \bar{x}_{\mathrm{exp}} - \bar{x}_{\mathrm{nat}}.
$$

It is tested with the member-block permutation test, which permutes whole members between the experiment and hist-nat, so it respects the correlation between a member's years. The change is significant where $p < 0.05$. (Notebook 03 also saves the pooled Welch t-test, `ttest`, which treats every year as independent and so calls more changes significant.)

### 1.1 What goes in

hist-nat's and `EXPERIMENT`'s ensemble means over the final 21 years, for `MODEL` in `SEASON`; the change between them; and the test's p-value.

In [ ]:
#(t): hist-nat's ensemble mean over the final WINDOW years, MODEL in SEASON
hist_nat_mean_da = final_mean_ds[VARIABLE].sel(model=MODEL, experiment='hist-nat', season=SEASON)
hist_nat_mean_da

In [ ]:
#(t): EXPERIMENT's
experiment_mean_da = final_mean_ds[VARIABLE].sel(model=MODEL, experiment=EXPERIMENT, season=SEASON)
experiment_mean_da

In [ ]:
#(t): The change (mean_block_ds['mean_change'] holds the same)
mean_change_da = experiment_mean_da - hist_nat_mean_da
mean_change_da

In [ ]:
#(t): Its p-value, from the member-block test
mean_pvalue_da = mean_block_ds['pvalue'].sel(model=MODEL, experiment=EXPERIMENT, season=SEASON)
mean_pvalue_da

In [ ]:
#(t): Not significant: p >= ALPHA (False where there is no data)
mean_not_significant_da = mean_pvalue_da >= ALPHA

In [ ]:
panels = fr.field_grid(
    [fr.Column(hist_nat_mean_da, scales.temperature, f'hist-nat\nfinal {WINDOW} years'),
     fr.Column(experiment_mean_da, scales.temperature, f'{EXPERIMENT}\nfinal {WINDOW} years'),
     fr.Column(mean_change_da, scales.change, f'Change\n{EXPERIMENT} − hist-nat', not_significant=mean_not_significant_da),
     fr.Column(mean_pvalue_da, scales.pvalue, 'p-value\nmember-block test')],
    title=f'What goes into the mean change: {MODEL}, {SEASON}',
    not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
)

### 1.2 One model

The change and its p-value for every experiment of `MODEL`.

In [ ]:
#(t): MODEL's change in every experiment, SEASON
model_mean_change_da = mean_block_ds['mean_change'].sel(model=MODEL, experiment=EXPERIMENTS, season=SEASON)
model_mean_change_da

In [ ]:
#(t): Its p-value
model_mean_pvalue_da = mean_block_ds['pvalue'].sel(model=MODEL, experiment=EXPERIMENTS, season=SEASON)
model_mean_pvalue_da

In [ ]:
#(t): Not significant
model_mean_not_significant_da = model_mean_pvalue_da >= ALPHA

In [ ]:
panels = fr.field_grid(
    [fr.Column(model_mean_change_da, scales.change, 'Change', not_significant=model_mean_not_significant_da),
     fr.Column(model_mean_pvalue_da, scales.pvalue, 'p-value')],
    title=f'{MODEL}: mean change, final {WINDOW} years against hist-nat, {SEASON}',
    row_dim='experiment',
    not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
)

### 1.3 Every model

In [ ]:
#(t): The change for every model and experiment, SEASON
season_mean_change_da = mean_block_ds['mean_change'].sel(experiment=EXPERIMENTS, season=SEASON)
season_mean_change_da

In [ ]:
#(t): Significant: p < ALPHA (False where there is no data)
season_mean_significant_da = mean_block_ds['pvalue'].sel(experiment=EXPERIMENTS, season=SEASON) < ALPHA

In [ ]:
#(t): Not significant, where there is data
season_mean_not_significant_da = ~season_mean_significant_da & season_mean_change_da.notnull()

In [ ]:
panels = fr.map_grid(
    season_mean_change_da, scales.change,
    title=f'Mean change, final {WINDOW} years against hist-nat, every model: {SEASON}',
    not_significant=season_mean_not_significant_da,
    not_significant_label=f'Not significant (member-block test, p ≥ {ALPHA:g})',
)

### 1.4 How many models

At each grid point, the number of models whose mean change is significant and positive (top row), and significant and negative (bottom row).

In [ ]:
#(t): How many models have a significant increase, and how many a significant decrease
mean_counts_da = sig.significant_counts(season_mean_change_da, season_mean_significant_da)
mean_counts_da

In [ ]:
panels = fr.count_grid(
    mean_counts_da, N_MODELS,
    title=f'How many models have a significant change in the mean: {SEASON}',
    label=f'Number of models with a significant decrease (blue) or increase (red), p < {ALPHA:g}',
    row_labels={'increase': 'Significant\nincrease', 'decrease': 'Significant\ndecrease'},
)

## 2. Signal to noise

S/N in the centre of the final 21 years (2003), the last year with a complete noise window, so it covers the same years as the tests (notebook 03, section 5):

- the **signal** $S$ is the experiment's forced response minus hist-nat's, where a forced response is the ensemble mean smoothed with LOWESS (81 years);
- the **noise** $N$ pools the two experiments' standard deviations $\sigma$ of each member about its forced response, over the 21 years centred on that year: $N = \sqrt{\tfrac{1}{2}\left(\sigma^2_{\mathrm{exp}} + \sigma^2_{\mathrm{nat}}\right)}$.

$|\mathrm{S/N}| \ge 2$ counts as emerged: the shift stands out from the year-to-year noise, so a single realisation, like the real world, would show it. The S/N colour scale is white where $|\mathrm{S/N}| < 2$, so it needs no dots.

### 2.1 What goes in

The forced responses of hist-nat and `EXPERIMENT` and the signal between them, then their noise and the pooled noise; `MODEL` in `SEASON`.

In [ ]:
#(t): The year shown: the centre of the final WINDOW years
sn_year = int(lesfmip_seasonal_sn_ds['year'][LAST_FULL_YEAR])
sn_year

In [ ]:
#(t): hist-nat's forced response in that year, MODEL in SEASON
hist_nat_forced_da = lesfmip_forced_ds[VARIABLE].sel(model=MODEL, experiment='hist-nat', season=SEASON, year=sn_year)
hist_nat_forced_da

In [ ]:
#(t): EXPERIMENT's
experiment_forced_da = lesfmip_forced_ds[VARIABLE].sel(model=MODEL, experiment=EXPERIMENT, season=SEASON, year=sn_year)
experiment_forced_da

In [ ]:
#(t): The signal (lesfmip_signal_ds holds the same)
signal_da = experiment_forced_da - hist_nat_forced_da
signal_da

In [ ]:
panels = fr.field_grid(
    [fr.Column(hist_nat_forced_da, scales.temperature, 'hist-nat\nforced response'),
     fr.Column(experiment_forced_da, scales.temperature, f'{EXPERIMENT}\nforced response'),
     fr.Column(signal_da, scales.change, f'Signal\n{EXPERIMENT} − hist-nat')],
    title=f'What goes into the signal: {MODEL}, {SEASON} {sn_year}',
)

In [ ]:
#(t): hist-nat's standard deviation about its forced response, in the WINDOW years centred on sn_year
hist_nat_sigma_da = lesfmip_experiment_noise_ds[VARIABLE].sel(model=MODEL, experiment='hist-nat', season=SEASON,
                                                              year=sn_year)
hist_nat_sigma_da

In [ ]:
#(t): EXPERIMENT's
experiment_sigma_da = lesfmip_experiment_noise_ds[VARIABLE].sel(model=MODEL, experiment=EXPERIMENT, season=SEASON,
                                                                year=sn_year)
experiment_sigma_da

In [ ]:
#(t): The noise: the two pooled (lesfmip_noise_ds holds the same)
noise_da = np.sqrt(0.5 * (hist_nat_sigma_da ** 2 + experiment_sigma_da ** 2))
noise_da

In [ ]:
panels = fr.field_grid(
    [fr.Column(hist_nat_sigma_da, scales.noise, 'hist-nat\nσ'),
     fr.Column(experiment_sigma_da, scales.noise, f'{EXPERIMENT}\nσ'),
     fr.Column(noise_da, scales.noise, 'Noise\nthe two pooled')],
    title=f'What goes into the noise: {MODEL}, {SEASON}, the {WINDOW} years centred on {sn_year}',
)

### 2.2 One model

The signal, the noise and S/N for every experiment of `MODEL`.

In [ ]:
#(t): MODEL's signal in every experiment, SEASON, sn_year
model_signal_da = lesfmip_signal_ds[VARIABLE].sel(model=MODEL, experiment=EXPERIMENTS, season=SEASON, year=sn_year)
model_signal_da

In [ ]:
#(t): Its noise
model_noise_da = lesfmip_noise_ds[VARIABLE].sel(model=MODEL, experiment=EXPERIMENTS, season=SEASON, year=sn_year)
model_noise_da

In [ ]:
#(t): S/N
model_sn_da = lesfmip_seasonal_sn_ds[VARIABLE].sel(model=MODEL, experiment=EXPERIMENTS, season=SEASON, year=sn_year)
model_sn_da

In [ ]:
panels = fr.field_grid(
    [fr.Column(model_signal_da, scales.change, 'Signal'),
     fr.Column(model_noise_da, scales.noise, 'Noise'),
     fr.Column(model_sn_da, scales.sn, 'S/N')],
    title=f'{MODEL}: signal, noise and S/N, {SEASON} {sn_year}',
    row_dim='experiment',
)

### 2.3 Every model

In [ ]:
#(t): S/N for every model and experiment, SEASON, sn_year
season_sn_da = lesfmip_seasonal_sn_ds[VARIABLE].sel(experiment=EXPERIMENTS, season=SEASON, year=sn_year)
season_sn_da

In [ ]:
panels = fr.map_grid(
    season_sn_da, scales.sn,
    title=f'S/N, every model: {SEASON} {sn_year} (white: |S/N| < {SN_THRESHOLD}, not emerged)',
)

### 2.4 How many models

The number of models where S/N ≥ 2 (top row) and S/N ≤ −2 (bottom row).

In [ ]:
#(t): Emerged: |S/N| >= SN_THRESHOLD (False where there is no data)
season_sn_emerged_da = np.abs(season_sn_da) >= SN_THRESHOLD

In [ ]:
#(t): How many models have emerged upwards, and how many downwards
sn_counts_da = sig.significant_counts(season_sn_da, season_sn_emerged_da)
sn_counts_da

In [ ]:
panels = fr.count_grid(
    sn_counts_da, N_MODELS,
    title=f'How many models have emerged from the noise: {SEASON} {sn_year}',
    label=f'Number of models with S/N ≤ −{SN_THRESHOLD} (blue) or S/N ≥ {SN_THRESHOLD} (red)',
    row_labels={'increase': f'S/N ≥ {SN_THRESHOLD}', 'decrease': f'S/N ≤ −{SN_THRESHOLD}'},
)

## 3. Quantile response: Q05, Q50 and Q95

How each part of the distribution has moved: the quantiles of each experiment's final 21 years (members and years pooled) minus hist-nat's over its whole record (notebook 03, section 3):

$$
\Delta Q_p = Q_p\left(x^{\mathrm{exp}}_{\text{final } L \text{ years}}\right) - Q_p\left(x^{\mathrm{nat}}_{\text{whole record}}\right).
$$

These are the values themselves, so the shift of the whole distribution is in every quantile. Equal changes in Q05, Q50 and Q95 are a pure shift; a bigger change in Q95 than in Q05 means the distribution has widened. There is no test of a single quantile: section 4 tests the width, Q95 − Q05.

### 3.1 What goes in

hist-nat's Q05, Q50 and Q95 over its whole record and `EXPERIMENT`'s over its final 21 years, `MODEL` in `SEASON`, and the change in each.

In [ ]:
#(t): hist-nat's quantiles over its whole record, MODEL in SEASON
hist_nat_q_da = q_histnat_base_ds[VARIABLE].sel(model=MODEL, season=SEASON, quantile=RESPONSE_QUANTILES)
hist_nat_q_da

In [ ]:
#(t): EXPERIMENT's over its final WINDOW years
experiment_q_da = final_q_da.sel(model=MODEL, experiment=EXPERIMENT, season=SEASON, quantile=RESPONSE_QUANTILES)
experiment_q_da

In [ ]:
#(t): The change in each (quantile_response_da holds the same)
quantile_change_da = experiment_q_da - hist_nat_q_da
quantile_change_da

In [ ]:
panels = fr.field_grid(
    [fr.Column(hist_nat_q_da, scales.temperature, 'hist-nat\nwhole record'),
     fr.Column(experiment_q_da, scales.temperature, f'{EXPERIMENT}\nfinal {WINDOW} years'),
     fr.Column(quantile_change_da, scales.change, f'Change\n{EXPERIMENT} − hist-nat')],
    title=f'What goes into the quantile response: {MODEL}, {SEASON}',
    row_dim='quantile',
    row_labels=QUANTILE_LABELS,
)

### 3.2 One model

The change in Q05, Q50 and Q95 for every experiment of `MODEL`.

In [ ]:
#(t): MODEL's change in each quantile, every experiment, SEASON
model_quantile_change_da = quantile_response_da.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASON)
model_quantile_change_da

In [ ]:
panels = fr.map_grid(
    model_quantile_change_da, scales.change,
    title=f'{MODEL}: change in Q05, Q50 and Q95, final {WINDOW} years against hist-nat, {SEASON}',
    row_dim='experiment', col_dim='quantile',
    col_labels={q: f'Δ{label}' for q, label in QUANTILE_LABELS.items()},
    panel_size=2.5,
)

### 3.3 Every model

The two ends of the distribution, Q05 and Q95 (the median moves with the mean, section 1).

In [ ]:
#(t): The change in Q05 for every model and experiment, SEASON
season_q05_change_da = quantile_response_da.sel(quantile=0.05, experiment=EXPERIMENTS, season=SEASON)
season_q05_change_da

In [ ]:
panels = fr.map_grid(
    season_q05_change_da, scales.change,
    title=f'Change in Q05, final {WINDOW} years against hist-nat, every model: {SEASON}',
)

In [ ]:
#(t): and in Q95
season_q95_change_da = quantile_response_da.sel(quantile=0.95, experiment=EXPERIMENTS, season=SEASON)
season_q95_change_da

In [ ]:
panels = fr.map_grid(
    season_q95_change_da, scales.change,
    title=f'Change in Q95, final {WINDOW} years against hist-nat, every model: {SEASON}',
)

### 3.4 Every model, every quantile

With no test there is nothing to count, so instead the change in every 5th percentile, averaged over the area (south of 40°S), one line per model. A flat line is a pure shift; a rising line means the upper quantiles moved more than the lower ones, so the distribution widened.

In [ ]:
#(t): The area-mean change in every 5th percentile, every model and experiment, SEASON
season_quantile_curve_da = quantile_curve_da.sel(experiment=EXPERIMENTS, season=SEASON)
season_quantile_curve_da

In [ ]:
panels = fr.quantile_curves(
    season_quantile_curve_da,
    title=f'Change at every 5th percentile, area mean south of {abs(config.LAT_MAX)}°S, every model: {SEASON}',
    units=UNITS,
)

## 4. Change in width, and which tail

The width is $W = Q_{95} - Q_{05}$ of the **internal variability**: each member minus its experiment's forced response (the ensemble median smoothed with LOWESS), so that a trend within the 21 years does not widen it. The change is the width of each experiment's final 21 years minus hist-nat's over its whole record (notebook 03, section 2):

$$
\Delta W = W\left(x'^{\,\mathrm{exp}}_{\text{final } L \text{ years}}\right) - W\left(x'^{\,\mathrm{nat}}_{\text{whole record}}\right).
$$

It is tested against 1000 hist-nat bootstrap trials, each the width of 10 hist-nat members over one 21-year window: significant where $p < 0.05$, i.e. where $\Delta W$ is outside the trials' central 95%.

**Which tail.** The median splits the width into the lower tail $Q_{50} - Q_{05}$ and the upper tail $Q_{95} - Q_{50}$. Their changes add up to $\Delta W$, so the larger one says which tail did the widening or narrowing; for temperature, the lower tail is the cold one. Only the width is tested.

### 4.1 What goes in

The two tails and the width of hist-nat over its whole record and of `EXPERIMENT` over its final 21 years, and the change in each; `MODEL` in `SEASON`.

In [ ]:
#(t): hist-nat's Q05, Q50 and Q95 of the internal variability, over its whole record, MODEL in SEASON
hist_nat_width_q_da = width_quantiles_ds['hist_nat'].sel(model=MODEL, season=SEASON)
hist_nat_width_q_da

In [ ]:
#(t): EXPERIMENT's, over its final WINDOW years
experiment_width_q_da = width_quantiles_ds['final'].sel(model=MODEL, experiment=EXPERIMENT, season=SEASON)
experiment_width_q_da

In [ ]:
#(t): hist-nat's lower tail Q50 - Q05, upper tail Q95 - Q50 and width Q95 - Q05, on a new 'part' dim
hist_nat_parts_da = rc.width_parts(hist_nat_width_q_da)
hist_nat_parts_da

In [ ]:
#(t): EXPERIMENT's
experiment_parts_da = rc.width_parts(experiment_width_q_da)
experiment_parts_da

In [ ]:
#(t): The change in each (tails_ds and qrange_change_ds hold the same)
parts_change_da = experiment_parts_da - hist_nat_parts_da
parts_change_da

In [ ]:
#(t): How the rows are labelled
PART_LABELS = {part: f'{part.capitalize()}\n{formula}' for part, formula in rc.WIDTH_PARTS.items()}
PART_LABELS

In [ ]:
panels = fr.field_grid(
    [fr.Column(hist_nat_parts_da, scales.spread, 'hist-nat\nwhole record'),
     fr.Column(experiment_parts_da, scales.spread, f'{EXPERIMENT}\nfinal {WINDOW} years'),
     fr.Column(parts_change_da, scales.spread_change, f'Change\n{EXPERIMENT} − hist-nat')],
    title=f'What goes into the change in width: {MODEL}, {SEASON} (internal variability)',
    row_dim='part',
    row_labels=PART_LABELS,
)

The test of that change: it is significant where it falls outside the central 95% of the hist-nat trials (the 2.5th to the 97.5th percentile of their changes).

In [ ]:
#(t): The width test for MODEL, EXPERIMENT and SEASON: the change, the trials' range, the p-value
width_test_ds = qrange_change_ds.sel(model=MODEL, experiment=EXPERIMENT, season=SEASON)
width_test_ds

In [ ]:
#(t): Not significant, where there is data
width_not_significant_da = ~width_test_ds['qrange_significant'] & width_test_ds['qrange_change'].notnull()

In [ ]:
panels = fr.field_grid(
    [fr.Column(width_test_ds['qrange_change'], scales.spread_change, 'Change in width', not_significant=width_not_significant_da),
     fr.Column(width_test_ds['null_lower'], scales.spread_change, 'hist-nat trials\n2.5th percentile'),
     fr.Column(width_test_ds['null_upper'], scales.spread_change, 'hist-nat trials\n97.5th percentile'),
     fr.Column(width_test_ds['qrange_pvalue'], scales.pvalue, 'p-value\nhist-nat bootstrap')],
    title=f'The width test: {MODEL}, {EXPERIMENT}, {SEASON}',
    not_significant_label=f'Not significant (inside the trials\' range, p ≥ {ALPHA:g})',
)

### 4.2 One model

For every experiment of `MODEL`: the change in each tail, the change in the width (their sum), and the width test's p-value.

In [ ]:
#(t): MODEL's tails, every experiment, SEASON
model_tails_ds = tails_ds.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASON)
model_tails_ds

In [ ]:
#(t): Its width test
model_width_test_ds = qrange_change_ds.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASON)
model_width_test_ds

In [ ]:
#(t): Not significant, where there is data
model_width_not_significant_da = ~model_width_test_ds['qrange_significant'] & model_width_test_ds['qrange_change'].notnull()

In [ ]:
panels = fr.field_grid(
    [fr.Column(model_tails_ds['lower_tail_change'], scales.spread_change, 'Lower tail\nΔ(Q50 − Q05)'),
     fr.Column(model_tails_ds['upper_tail_change'], scales.spread_change, 'Upper tail\nΔ(Q95 − Q50)'),
     fr.Column(model_width_test_ds['qrange_change'], scales.spread_change, 'Width\nΔ(Q95 − Q05)',
               not_significant=model_width_not_significant_da),
     fr.Column(model_width_test_ds['qrange_pvalue'], scales.pvalue, 'p-value\nwidth test')],
    title=f'{MODEL}: change in the tails and the width, final {WINDOW} years against hist-nat, {SEASON}',
    row_dim='experiment',
    not_significant_label=f'Width change not significant (p ≥ {ALPHA:g})',
)

### 4.3 Every model

In [ ]:
#(t): The change in width for every model and experiment, SEASON
season_width_change_da = qrange_change_ds['qrange_change'].sel(experiment=EXPERIMENTS, season=SEASON)
season_width_change_da

In [ ]:
#(t): Significant (p < ALPHA in notebook 03)
season_width_significant_da = qrange_change_ds['qrange_significant'].sel(experiment=EXPERIMENTS, season=SEASON)

In [ ]:
#(t): Not significant, where there is data
season_width_not_significant_da = ~season_width_significant_da & season_width_change_da.notnull()

In [ ]:
panels = fr.map_grid(
    season_width_change_da, scales.spread_change,
    title=f'Change in width Q95 − Q05, final {WINDOW} years against hist-nat, every model: {SEASON}',
    not_significant=season_width_not_significant_da,
    not_significant_label=f'Not significant (hist-nat bootstrap, p ≥ {ALPHA:g})',
)

### 4.4 How many models

The number of models with a significantly wider distribution (top row) and a significantly narrower one (bottom row).

In [ ]:
#(t): How many models are significantly wider, and how many narrower
width_counts_da = sig.significant_counts(season_width_change_da, season_width_significant_da)
width_counts_da

In [ ]:
panels = fr.count_grid(
    width_counts_da, N_MODELS,
    title=f'How many models have a significant change in width: {SEASON}',
    label=f'Number of models significantly narrower (blue) or wider (red), p < {ALPHA:g}',
    row_labels={'increase': 'Significantly\nwider', 'decrease': 'Significantly\nnarrower'},
)

### 4.5 Which tail, every model

The upper tail's change minus the lower tail's: red where the upper (warm) tail stretched more or shrank less, blue where the lower (cold) tail did. Dotted where the width change itself is not significant.

In [ ]:
#(t): The tail asymmetry for every model and experiment, SEASON
season_tail_asymmetry_da = tails_ds['tail_asymmetry'].sel(experiment=EXPERIMENTS, season=SEASON)
season_tail_asymmetry_da

In [ ]:
panels = fr.map_grid(
    season_tail_asymmetry_da, scales.tail,
    title=f'Which tail changed more: upper-tail minus lower-tail change, every model: {SEASON}',
    not_significant=season_width_not_significant_da,
    not_significant_label=f'Width change not significant (p ≥ {ALPHA:g})',
)

## 5. The whole distribution: where the mean and the width have changed

Sections 1, 3 and 4 together, in the form of Bracegirdle et al. (2024, Fig. 1): the low end of the distribution (Q05), its middle (the mean), its high end (Q95) and its width (Q95 − Q05), for hist-nat and an experiment, and the change in each. Equal changes in Q05, the mean and Q95 are a pure shift; a bigger change in Q05 than in Q95 is a distribution narrowing from below.

Then each grid point is put in one of nine classes: the mean significantly down, unchanged or up (section 1's test), times the width significantly narrower, unchanged or wider (section 4's test). The 3 × 3 key reads like a table: red is a warmer mean, blue a cooler one; yellow mixed in is a narrower distribution, purple a wider one.

`MEAN_TEST` chooses the test for the mean: `'significant'` is the member-block test, as in section 1; `'robust'` asks for $|\mathrm{S/N}| \ge 2$ as well (section 2), and `'emerged'` for that alone.

In [ ]:
#(t): Which test says the mean has changed: 'significant' (member-block, section 1), 'emerged' (S/N) or 'robust' (both)
MEAN_TEST = 'significant'

In [ ]:
#(t): S/N in the year of section 2, for the 'emerged' and 'robust' tests
sn_final_da = lesfmip_seasonal_sn_ds[VARIABLE].sel(year=sn_year)
sn_final_da

In [ ]:
#(t): Every change and its test in one Dataset on (model, experiment, season, lat, lon)
change_ds = rc.change_summary(mean_block_ds['mean_change'], mean_block_ds['pvalue'], sn_final_da, qrange_change_ds,
                              tails_ds, alpha=ALPHA, sn_threshold=SN_THRESHOLD)
change_ds

In [ ]:
#(t): hist-nat's distribution: Q05 and Q95 over its whole record, the mean over the final WINDOW years, and the
#(t): width of its internal variability (as tested in section 4)
hist_nat_fields_ds = xr.Dataset({
    'Q05': q_histnat_base_ds[VARIABLE].sel(quantile=0.05, drop=True),
    'Mean': final_mean_ds[VARIABLE].sel(experiment='hist-nat', drop=True),
    'Q95': q_histnat_base_ds[VARIABLE].sel(quantile=0.95, drop=True),
    'Q95 − Q05': qrange_change_ds['hist_nat_qrange'],
})
hist_nat_fields_ds

In [ ]:
#(t): Every experiment's, over its final WINDOW years
experiment_fields_ds = xr.Dataset({
    'Q05': final_q_da.sel(quantile=0.05, drop=True),
    'Mean': final_mean_ds[VARIABLE].drop_sel(experiment='hist-nat'),
    'Q95': final_q_da.sel(quantile=0.95, drop=True),
    'Q95 − Q05': qrange_change_ds['experiment_qrange'],
})
experiment_fields_ds

In [ ]:
#(t): The change in each: the same as sections 1, 3 and 4
change_fields_ds = experiment_fields_ds - hist_nat_fields_ds
change_fields_ds

In [ ]:
#(t): Which changes are significant: the mean by MEAN_TEST, the width by the hist-nat bootstrap (Q05 and Q95 have no test)
change_significant_ds = xr.Dataset({'Mean': change_ds[f'mean_{MEAN_TEST}'], 'Q95 − Q05': change_ds['width_significant']})
change_significant_ds

In [ ]:
#(t): Not significant, where there is data
change_not_significant_ds = ~change_significant_ds & change_fields_ds[list(change_significant_ds)].notnull()

In [ ]:
#(t): The sign of each significant change: +1 up, -1 down, 0 not significant
mean_sign_da = sig.significant_sign(change_fields_ds['Mean'], change_significant_ds['Mean'])
width_sign_da = sig.significant_sign(change_fields_ds['Q95 − Q05'], change_significant_ds['Q95 − Q05'])

In [ ]:
#(t): The joint class, 0 to 8, in the order of the 3 x 3 key
joint_class_da = rc.joint_class(mean_sign_da, width_sign_da)
joint_class_da

### 5.1 What goes in

The distribution of hist-nat and of `EXPERIMENT`, `MODEL` in `SEASON`, and the change in each part. The top two rows share colour scales, and so does the bottom row; the width has its own. hist-nat's mean is over the same final 21 years as the experiment's (as the mean test compares them), its quantiles over its whole record (as the width test does).

In [ ]:
#(t): MODEL in SEASON: hist-nat
model_hist_nat_fields_ds = hist_nat_fields_ds.sel(model=MODEL, season=SEASON)
model_hist_nat_fields_ds

In [ ]:
#(t): EXPERIMENT
model_experiment_fields_ds = experiment_fields_ds.sel(model=MODEL, experiment=EXPERIMENT, season=SEASON)

In [ ]:
#(t): The change
model_change_fields_ds = change_fields_ds.sel(model=MODEL, experiment=EXPERIMENT, season=SEASON)

In [ ]:
#(t): Where the change is not significant
model_change_not_significant_ds = change_not_significant_ds.sel(model=MODEL, experiment=EXPERIMENT, season=SEASON)

In [ ]:
panels = fr.block_grid(
    [fr.Block({'hist-nat': model_hist_nat_fields_ds, EXPERIMENT: model_experiment_fields_ds},
              scales=[scales.temperature, scales.temperature, scales.temperature, scales.spread]),
     fr.Block({'Change': model_change_fields_ds},
              scales=[scales.change, scales.change, scales.change, scales.spread_change],
              not_significant={'Change': model_change_not_significant_ds})],
    title=f'The distribution and its change: {MODEL}, {EXPERIMENT}, {SEASON} (final {WINDOW} years against hist-nat)',
    tag=True,
    not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
)

The mean change and the width change with their tests, and the class they give.

In [ ]:
#(t): MODEL's joint class, EXPERIMENT in SEASON
model_joint_class_da = joint_class_da.sel(model=MODEL, experiment=EXPERIMENT, season=SEASON)

In [ ]:
panels = fr.field_grid(
    [fr.Column(model_change_fields_ds['Mean'], scales.change, 'Mean change',
               not_significant=model_change_not_significant_ds['Mean']),
     fr.Column(model_change_fields_ds['Q95 − Q05'], scales.spread_change, 'Change in width',
               not_significant=model_change_not_significant_ds['Q95 − Q05']),
     fr.Column(model_joint_class_da, scales.joint, 'What changed')],
    title=f'Where the mean and the width have changed: {MODEL}, {EXPERIMENT}, {SEASON}',
    not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
)

### 5.2 One model

The change in every part of the distribution, and what changed, for every experiment of `MODEL`: the bottom row of 5.1 for each forcing.

In [ ]:
#(t): MODEL, every experiment, SEASON: the changes
model_forcing_changes_ds = change_fields_ds.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASON)
model_forcing_changes_ds

In [ ]:
#(t): where they are not significant
model_forcing_not_significant_ds = change_not_significant_ds.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASON)

In [ ]:
#(t): and the joint class
model_forcing_class_da = joint_class_da.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASON)

In [ ]:
panels = fr.field_grid(
    [fr.Column(model_forcing_changes_ds['Q05'], scales.change, 'ΔQ05'),
     fr.Column(model_forcing_changes_ds['Mean'], scales.change, 'Δ mean',
               not_significant=model_forcing_not_significant_ds['Mean']),
     fr.Column(model_forcing_changes_ds['Q95'], scales.change, 'ΔQ95'),
     fr.Column(model_forcing_changes_ds['Q95 − Q05'], scales.spread_change, 'Δ(Q95 − Q05)',
               not_significant=model_forcing_not_significant_ds['Q95 − Q05']),
     fr.Column(model_forcing_class_da, scales.joint, 'What changed')],
    title=f'{MODEL}: how each forcing changes the distribution, {SEASON} (final {WINDOW} years against hist-nat)',
    row_dim='experiment',
    tag=True,
    not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
)

### 5.3 Every model

The same for `EXPERIMENT`, one row per model; then the joint class of every model and experiment.

In [ ]:
#(t): Every model, EXPERIMENT in SEASON: the changes
experiment_model_changes_ds = change_fields_ds.sel(experiment=EXPERIMENT, season=SEASON)
experiment_model_changes_ds

In [ ]:
#(t): where they are not significant
experiment_model_not_significant_ds = change_not_significant_ds.sel(experiment=EXPERIMENT, season=SEASON)

In [ ]:
#(t): and the joint class
experiment_model_class_da = joint_class_da.sel(experiment=EXPERIMENT, season=SEASON)

In [ ]:
panels = fr.field_grid(
    [fr.Column(experiment_model_changes_ds['Q05'], scales.change, 'ΔQ05'),
     fr.Column(experiment_model_changes_ds['Mean'], scales.change, 'Δ mean',
               not_significant=experiment_model_not_significant_ds['Mean']),
     fr.Column(experiment_model_changes_ds['Q95'], scales.change, 'ΔQ95'),
     fr.Column(experiment_model_changes_ds['Q95 − Q05'], scales.spread_change, 'Δ(Q95 − Q05)',
               not_significant=experiment_model_not_significant_ds['Q95 − Q05']),
     fr.Column(experiment_model_class_da, scales.joint, 'What changed')],
    title=f'Every model: how {EXPERIMENT} changes the distribution, {SEASON} (final {WINDOW} years against hist-nat)',
    row_dim='model',
    panel_size=2.3,
    not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
)

In [ ]:
#(t): The joint class for every model and experiment, SEASON
season_class_da = joint_class_da.sel(experiment=EXPERIMENTS, season=SEASON)
season_class_da

In [ ]:
panels = fr.map_grid(
    season_class_da, scales.joint,
    title=f'Where the mean and the width have changed, every model: {SEASON}',
)

### 5.4 How many models

The number of models where the mean has changed and the distribution has widened (top row), or narrowed (bottom row).

In [ ]:
#(t): How many models changed in the mean and became wider, and how many narrower
joint_counts_da = rc.joint_change_counts(change_ds, mean_test=MEAN_TEST)
joint_counts_da

In [ ]:
#(t): SEASON
season_joint_counts_da = joint_counts_da.sel(experiment=EXPERIMENTS, season=SEASON)

In [ ]:
panels = fr.count_grid(
    season_joint_counts_da, N_MODELS,
    title=f'How many models have changed in both the mean and the width: {SEASON}',
    label='Number of models where the mean changed and the distribution narrowed (blue) or widened (red)',
    row_dim='change',
    row_labels={'mean and wider': 'Mean and\nwider', 'mean and narrower': 'Mean and\nnarrower'},
)

### 5.5 A closer look at one point

At a single grid point, the final years of each experiment (members pooled) against hist-nat's:

- **a)** the two distributions, with arrows from each hist-nat quantile (Q05, Q50, Q95) to the experiment's. Equal arrows are a pure shift;
- **b)** the change in every quantile, for every experiment, with a bootstrap (whole members) 5–95% range. A flat curve is a pure shift; a rising one, the upper tail moving more; a falling one, the lower tail.

The point is where the mean and the width both changed and the width changed most (south of 60°S). Set `change_point` to look anywhere else.

In [ ]:
#(t): The point: where the mean and the width both changed, and the width changed most
change_point = rc.strongest_joint_change(change_ds, MODEL, EXPERIMENT, SEASON, mean_test=MEAN_TEST)
# change_point = dict(lat=-75.0, lon=-180.0)   # or choose one
change_point

In [ ]:
#(t): The experiments MODEL has, hist-nat first, then in the order of the figures
point_experiments = [e for e in ['hist-nat', *EXPERIMENTS] if e in lesfmip_season_tree[MODEL].children]
point_experiments

In [ ]:
#(t): Their final years at that point, on (member, year)
point_samples = rc.final_years(lesfmip_season_tree, MODEL, point_experiments, change_point, SEASON, years=WINDOW,
                               variable=VARIABLE)
point_samples[EXPERIMENT]

In [ ]:
#(t): The change in every quantile of each experiment against hist-nat, with its bootstrap range
point_shifts = {e: rc.quantile_shift(point_samples[e], point_samples['hist-nat']) for e in point_experiments if e != 'hist-nat'}
point_shifts[EXPERIMENT]

In [ ]:
#(t): Quick look: hist-nat's and EXPERIMENT's final years (members pooled), and every experiment's quantile shift
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name in ('hist-nat', EXPERIMENT):
    axes[0].hist(point_samples[name].values.ravel(), bins=30, alpha=0.5, label=name)
for name, shift_ds in point_shifts.items():
    shift_ds['shift'].plot(x='quantile', ax=axes[1], label=name)
axes[0].set_title('point_samples')
axes[1].set_title("point_shifts[experiment]['shift']")
axes[0].legend()
axes[1].legend();

In [ ]:
panels = rcp.shift_and_widen(
    point_samples, point_shifts, EXPERIMENT,
    test=change_ds.sel(model=MODEL, experiment=EXPERIMENT, season=SEASON).sel(change_point),
    units=UNITS,
    title=f'How the distribution changes: {MODEL}, {SEASON} at {formatting.format_latlon(change_point)}',
)

### 5.6 Every model and forcing, as area means

Area means south of 60°S. First every model's change in the mean and in the width for each forcing (dots: one per model; bars: the multi-model median), then the two against each other, then the lower tail's change against the upper tail's (above the dashed 1:1 line the upper tail stretched more).

In [ ]:
#(t): Area means south of 60°S of every change, and the share of the area in each class
change_regional_ds = rc.regional_mean(change_ds.sel(experiment=EXPERIMENTS), lat_max=-60, mean_test=MEAN_TEST)
change_regional_ds

In [ ]:
panels = fr.forcing_dots(
    change_regional_ds,
    {'mean_change': f'Mean change ({UNITS})', 'width_change': f'Change in Q95 − Q05 ({UNITS})'},
    title='Area mean south of 60°S: every model (markers) and the multi-model median (bars)',
    columns=SEASONS, experiments=EXPERIMENTS,
)

In [ ]:
panels = rcp.response_scatter(change_regional_ds, 'mean_change', 'width_change',
                              title='Area mean south of 60°S: change in width against mean change')

In [ ]:
panels = rcp.response_scatter(change_regional_ds, 'lower_tail_change', 'upper_tail_change', diagonal=True,
                              title='Area mean south of 60°S: upper-tail against lower-tail change')

In [ ]:
#(t): The share of the area south of 60°S where neither, the mean only, the width only, or both changed, SEASON (%)
class_columns = [f'fraction_{name.replace(" ", "_")}' for name in rc.CHANGE_CLASSES.values()]
class_share_df = (100 * change_regional_ds[class_columns].sel(season=SEASON)).to_dataframe().round(0)
class_share_df

## 6. Additivity: does historical equal the sum of the single forcings?

Each experiment's change from its own pre-industrial baseline: its final 21 years minus its own 1850–1900 (notebook 03, section 6). A single-forcing experiment contains *only* its forcing, so this is its response; measured from hist-nat, as in the other sections, a sum of four forcings would subtract hist-nat's volcanic cooling four times.

$$
\Delta_e = \bar{x}^{\,e}_{\text{final } L \text{ years}} - \bar{x}^{\,e}_{1850\text{–}1900},
\qquad
\text{residual} = \Delta_{\mathrm{historical}} - \left(\Delta_{\mathrm{nat}} + \Delta_{\mathrm{GHG}} + \Delta_{\mathrm{aer}} + \Delta_{\mathrm{O_3}}\right).
$$

The residual holds everything the parts leave out: land use (hist-lu, which should be very small over Antarctica), any other forcing, and non-additive interaction between forcings. There is no test, so the last part is a table rather than a count.

In [ ]:
#(t): Every term: each part, their sum, historical and the residual; and each part's fraction of historical
additivity_ds = rc.additivity(own_changes_da)
additivity_ds

### 6.1 What goes in

Each experiment's change from its own baseline, their sum, historical's, and the residual; `MODEL` in `SEASON`.

In [ ]:
#(t): MODEL's terms, SEASON
model_terms_da = additivity_ds['response'].sel(model=MODEL, season=SEASON)
model_terms_da

In [ ]:
panels = fr.map_grid(
    model_terms_da, scales.change,
    title=f'What goes into the additivity: {MODEL}, {SEASON}, final {WINDOW} years minus 1850–1900',
    row_dim=None, col_dim='term',
)

### 6.2 One model

The sum of the parts, historical, and the residual for `MODEL`, every season.

In [ ]:
#(t): MODEL's sum of parts, historical and residual, every season
model_additivity_da = additivity_ds['response'].sel(model=MODEL, season=SEASONS,
                                                    term=['sum of parts', 'historical', 'residual'])
model_additivity_da

In [ ]:
panels = fr.map_grid(
    model_additivity_da, scales.change,
    title=f'{MODEL}: historical against the sum of the single forcings',
    row_dim='season', col_dim='term', panel_size=2.5,
)

### 6.3 Every model

In [ ]:
#(t): Every model's sum of parts, historical and residual, SEASON
season_additivity_da = additivity_ds['response'].sel(season=SEASON, term=['sum of parts', 'historical', 'residual'])
season_additivity_da

In [ ]:
panels = fr.map_grid(
    season_additivity_da, scales.change,
    title=f'Historical against the sum of the single forcings, every model: {SEASON}',
    row_dim='model', col_dim='term',
)

### 6.4 Every model, as numbers

Area means south of 60°S of every term, and each part's share of historical's area mean (the share of the area means, which does not blow up where historical's change is small at a point).

In [ ]:
#(t): Area means south of 60°S of every term, SEASON
antarctic_terms_da = sig.area_mean(additivity_ds['response'].sel(season=SEASON, lat=slice(None, -60)))
antarctic_terms_da

In [ ]:
#(t): As a table: models down, terms across
antarctic_terms_df = antarctic_terms_da.to_series().unstack('term')[list(additivity_ds['term'].values)].round(2)
antarctic_terms_df

In [ ]:
#(t): Each part's and the residual's share of historical (%)
share_columns = [*rc.ADDITIVE_PARTS, 'residual']
share_of_historical_df = (100 * antarctic_terms_df[share_columns].div(antarctic_terms_df['historical'], axis=0)).round(0)
share_of_historical_df

## 7. Zonal means and the sea-ice edge

After Bracegirdle et al. (2024, *npj Clim. Atmos. Sci.* 7, 276; [doi:10.1038/s41612-024-00822-y](https://doi.org/10.1038/s41612-024-00822-y)), Figs 1–3, 6 and 8, which compare the change in extreme seasons with the change in the background climate, latitude by latitude, and relate the difference to the retreating sea ice. Their methods differ from ours, so these figures keep their layout but show this project's changes:

| Column | Here | The paper |
|---|---|---|
| Mean | $\Delta$ mean (section 1) | the background climate (smoothed ensemble mean) |
| Low extremes | $\Delta(Q_{05} - Q_{50})$ of the internal variability: > 0 where the low extremes move up towards the middle | p10, minus the background |
| High extremes | $\Delta(Q_{95} - Q_{50})$: > 0 where the high extremes move away from the middle | p90, minus the background |
| Width | $\Delta(Q_{95} - Q_{05})$ = high − low (section 4) | p90 − p10 |

**Every model is shown on its own** (the paper averages five). The zonal means are tested (notebook 03, section 8): dots on a line mark a significant change in the mean (member-block test) or the width (hist-nat bootstrap), and the grey band is what hist-nat's own variability gives the width. The dashed lines are the sea-ice edge (equivalent latitude, notebook 01 section 5) in each experiment's final years.

The paper's key result for temperature: in winter, over the retreating sea ice, the cold extremes warm much more than the warm extremes, so the distribution narrows. Here that would show as a positive low-extreme change and a negative width change just poleward of the ice edge, in the experiments whose edge has moved (set `SEASON = 'JJA'` to see winter).

In [ ]:
#(t): The four columns, zonally averaged, with the zonal tests
zonal_ds = zonal.zonal_summary(change_ds, zonal_mean_ds, zonal_width_ds)
zonal_ds

### 7.1 What goes in

The maps behind the zonal means for `MODEL` and `EXPERIMENT` (the paper's Figs 1 and 3), with the 15% ice edge of the experiment (solid) and of hist-nat (dashed). Dotted where the mean or the width change is not significant.

In [ ]:
panels = rcp.extreme_change_maps(
    change_ds, MODEL, EXPERIMENT, seasons=[SEASON], mean_test=MEAN_TEST, ice=ice_climatology_da, units=UNITS,
    mean_levels=scales.change.levels, extreme_levels=scales.spread_change.levels,
    title=f'What goes into the zonal means: {MODEL}, {EXPERIMENT}, {SEASON}',
)

### 7.2 One model

`MODEL`'s change at each latitude, every experiment as a line.

In [ ]:
#(t): Quick look: the four columns for MODEL in SEASON against latitude, one line per experiment
model_zonal_ds = zonal_ds.sel(model=MODEL, season=SEASON)
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
for ax, name in zip(axes, ['mean_change', 'low_extreme_change', 'high_extreme_change', 'width_change']):
    model_zonal_ds[name].plot.line(x='lat', hue='experiment', ax=ax)
    ax.set_title(name)

In [ ]:
fig, axes = zp.zonal_change_grid(zonal_ds, SEASON, row_values=[MODEL], edges=ice_edges_da, units=UNITS,
                                 title=f'{MODEL}: zonal-mean change, final {WINDOW} years against hist-nat, {SEASON}')

### 7.3 Every model

One row per model, the experiments as lines. The mean has its own scale; the extremes and the width share one, so the width column is the high column minus the low.

In [ ]:
fig, axes = zp.zonal_change_grid(zonal_ds, SEASON, edges=ice_edges_da, units=UNITS,
                                 title=f'Zonal-mean change, final {WINDOW} years against hist-nat, every model: {SEASON}')

### 7.4 Every model, by forcing

The same the other way round: one row per experiment and the models as lines, to see how far the models agree on each forcing.

In [ ]:
fig, axes = zp.zonal_change_grid(zonal_ds, SEASON, rows='experiment', lines='model', edges=ice_edges_da, units=UNITS,
                                 title=f'Zonal-mean change by forcing, every model as a line: {SEASON}')

### 7.5 Where the seasons vary most

hist-nat's own width $Q_{95} - Q_{05}$ (whole record), zonally averaged, with each model's hist-nat ice edge. The paper finds the year-to-year variability of winter temperature largest just poleward of the ice edge, where a season can be over ice or open water; a model whose peak sits elsewhere, or whose edge is far from the others', will put its changes in variability at other latitudes too.

In [ ]:
fig, axes = zp.zonal_reference_width(zonal_ds, seasons=[SEASON], edges=ice_edges_da, units=UNITS,
                                     title=f'Year-to-year variability in hist-nat: zonal-mean width, {SEASON}')

### 7.6 The sea ice itself (JASMIN)

Each experiment's extent through time (ensemble mean and the 5–95% of members), and the equivalent latitude of each model's edge in the final years.

In [ ]:
#(t): Each experiment's extent: the ensemble mean and the 5-95% of members
ice_extent_summary_ds = sea_ice.extent_summary(ice_extents_tree)
ice_extent_summary_ds

In [ ]:
fig, axes = sip.extent_timeseries(ice_extent_summary_ds, seasons=[SEASON],
                                  title=f'Sea-ice extent, every model and experiment: {SEASON}')

In [ ]:
#(t): The equivalent latitude of each model's ice edge in the final years
ice_edges_da.to_dataframe().unstack('season')['edge_latitude'].round(1)